# TRIAGE-EG Completion v1.1 — 37 Repair Multimodal Evidence

Fail-closed recovery of ASR, XCLIP, Grounding-DINO and isolated PP-OCR. No GT is mounted or read. Empty ASR/XCLIP/DINO/OCR channels stop the run. Output is `/kaggle/working/triage_eg_completion_v11_evidence_bundle.zip`.

In [ ]:
import os
from pathlib import Path
REPO_URL="https://github.com/Irthn1311/AIC2026_TeamPTK_SGU.git"
REPO_REF="TRIAGEEG"
ANCHOR="d338d8d809bbb9e057e18becc607af2eb3bea254"
REPO_DIR=Path(os.environ.get("AIC_REPO_DIR","/kaggle/working/AIC2026_TeamPTK_SGU"))
RAW_INPUT=Path(os.environ.get("AIC_DATA_ROOT","/kaggle/input/datasets/nadkli/dataset-aic"))
TEAM_INPUT=Path(os.environ.get("AIC_TEAM_EVAL_ROOT","/kaggle/input/datasets/irthn1311/aic2026-team-eval-dev-v1"))
FREEZE_INPUT=Path(os.environ.get("AIC_FS1_MASTER_FREEZE_ROOT","/kaggle/input/datasets/irthn1311/fs1-master-preparation-freeze-2026-08-18"))
DIAGNOSIS_INPUT=Path(os.environ.get("AIC_COMPLETION_DIAGNOSIS_ROOT","/kaggle/input/datasets/irthn1311/triageeg-completion-diagnosis-freeze-2026-08-19"))
WHISPER_INPUT=Path(os.environ.get("AIC_WHISPER_ROOT","/kaggle/input/datasets/irthn1311/fs1-whisper-large-v3-turbo-asset"))
QWEN_INPUT=Path(os.environ.get("AIC_QWEN_ROOT","/kaggle/input/datasets/irthn1311/fs1-qwen2-5-vl-3b-instruct-asset"))
OUTPUT_ROOT=Path("/kaggle/working/triage_eg_completion_v11_evidence")
OUTPUT_ZIP=Path("/kaggle/working/triage_eg_completion_v11_evidence_bundle.zip")
OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
print({"required_inputs":{"raw":str(RAW_INPUT),"team_eval_queries":str(TEAM_INPUT),"fs1_master_freeze":str(FREEZE_INPUT),"completion_diagnosis_freeze":str(DIAGNOSIS_INPUT),"whisper_asset":str(WHISPER_INPUT),"qwen_asset":str(QWEN_INPUT)},"internet_required":"YES_FOR_PINNED_XCLIP_DINO_AND_ISOLATED_OCR","gpu":"TESLA_T4","output_zip":str(OUTPUT_ZIP)})


In [ ]:
import subprocess,sys
if not (REPO_DIR/".git").is_dir():
    subprocess.run(["git","clone","--branch",REPO_REF,"--single-branch",REPO_URL,str(REPO_DIR)],check=True)
subprocess.run(["git","fetch","origin",REPO_REF],cwd=REPO_DIR,check=True)
subprocess.run(["git","checkout","--detach","FETCH_HEAD"],cwd=REPO_DIR,check=True)
HEAD=subprocess.check_output(["git","rev-parse","HEAD"],cwd=REPO_DIR,text=True).strip()
ancestor=subprocess.run(["git","merge-base","--is-ancestor",ANCHOR,HEAD],cwd=REPO_DIR).returncode==0
if not ancestor: raise RuntimeError(f"COMPLETION_V11_LINEAGE_FAIL anchor={ANCHOR} HEAD={HEAD}")
sys.path.insert(0,str(REPO_DIR/"src"))
import torch
GPU={"torch":torch.__version__,"cuda_build":torch.version.cuda,"available":torch.cuda.is_available(),"name":torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}
if not torch.cuda.is_available() or "T4" not in str(GPU["name"]).upper(): raise RuntimeError(f"COMPLETION_V11_T4_REQUIRED: {GPU}")
print({"HEAD":HEAD,"anchor_is_ancestor":ancestor,"gpu":GPU})


In [ ]:
import json,zipfile
def bounded(root,name,max_depth=6):
    output=[]; root=Path(root)
    if not root.exists(): return output
    for directory,subdirs,files in os.walk(root):
        current=Path(directory); depth=len(current.relative_to(root).parts)
        subdirs[:]=[] if depth>=max_depth else [x for x in subdirs if x not in {".cache","blobs","snapshots"}]
        if name in files: output.append(current/name)
    return output
def unique(values,label):
    values=sorted(set(Path(x).resolve() for x in values))
    if len(values)!=1: raise RuntimeError(f"Expected one {label}; found {values}")
    return values[0]
def find_any(hint,name):
    return unique(bounded(hint,name) or bounded("/kaggle/input",name),name)
def model_root(hint):
    return find_any(hint,"config.json").parent
WHISPER_ROOT=model_root(WHISPER_INPUT); QWEN_ROOT=model_root(QWEN_INPUT)
TEAM_QUERIES=bounded(TEAM_INPUT,"queries.jsonl") or bounded("/kaggle/input","queries.jsonl")
BENCH={name:unique([p.parent for p in TEAM_QUERIES if p.parent.name==name],name) for name in ("dev_cross_60","dev_l21_150")}
PROTOCOL=find_any(FREEZE_INPUT,"FS1_PROTOCOL.md"); PREP=PROTOCOL.parent
diagnosis_matches=bounded(DIAGNOSIS_INPUT,"DIAGNOSIS.json")
if diagnosis_matches: DIAGNOSIS_PATH=unique(diagnosis_matches,"DIAGNOSIS.json")
else:
    diagnosis_zip=find_any(DIAGNOSIS_INPUT,"TRIAGEEG_COMPLETION_DIAGNOSIS_FREEZE_2026-08-19.zip"); diagnosis_extract=Path("/kaggle/working/completion_diagnosis_freeze"); diagnosis_extract.mkdir(parents=True,exist_ok=True); zipfile.ZipFile(diagnosis_zip).extractall(diagnosis_extract); DIAGNOSIS_PATH=unique(bounded(diagnosis_extract,"DIAGNOSIS.json"),"DIAGNOSIS.json")
DIAGNOSIS=json.loads(DIAGNOSIS_PATH.read_text())
if DIAGNOSIS.get("repo_head")!=ANCHOR or set(DIAGNOSIS.get("confirmed_failures",{}))!={"asr","xclip","ppocr","grounding_dino","event_graph"}: raise RuntimeError("DIAGNOSIS_FREEZE_CONTRACT_MISMATCH")
print({"whisper":str(WHISPER_ROOT),"qwen":str(QWEN_ROOT),"benchmarks":{k:str(v) for k,v in BENCH.items()},"freeze":str(PREP),"diagnosis":str(DIAGNOSIS_PATH)})


In [ ]:
test_env=os.environ.copy(); test_env["PYTHONPATH"]=str(REPO_DIR/"src")+(os.pathsep+test_env["PYTHONPATH"] if test_env.get("PYTHONPATH") else "")
test=subprocess.run([sys.executable,"-m","pytest","tests/unit/fs1_v11","tests/unit/fs1","tests/unit/bcf1_protected_late_fusion","-q"],cwd=REPO_DIR,env=test_env,capture_output=True,text=True)
TEST_SUMMARY={"returncode":test.returncode,"stdout_tail":test.stdout.splitlines()[-20:],"stderr_tail":test.stderr.splitlines()[-20:]}
if test.returncode: raise RuntimeError(TEST_SUMMARY)
print(TEST_SUMMARY)


In [ ]:
from huggingface_hub import snapshot_download
from triage_eg.fs1_v11.contracts import XCLIP_ID,XCLIP_REVISION,DINO_ID,DINO_REVISION
XCLIP_ROOT=Path(snapshot_download(XCLIP_ID,revision=XCLIP_REVISION,local_dir="/kaggle/working/completion_assets/xclip"))
DINO_ROOT=Path(snapshot_download(DINO_ID,revision=DINO_REVISION,local_dir="/kaggle/working/completion_assets/dino"))
print({"xclip":str(XCLIP_ROOT),"dino":str(DINO_ROOT)})


In [ ]:
# Mandatory ASR: MP4 is decoded to mono 16k float PCM by ffmpeg before Whisper.
from transformers import AutoModelForSpeechSeq2Seq,AutoProcessor,pipeline
from triage_eg.fs1_v11.asr import lexical_index,transcribe_video
from triage_eg.data.stage0_audit.asset_resolver import discover_layout,resolve_assets
video_parts,keyframe_parts=discover_layout(RAW_INPUT)
processor=AutoProcessor.from_pretrained(WHISPER_ROOT,local_files_only=True)
model=AutoModelForSpeechSeq2Seq.from_pretrained(WHISPER_ROOT,local_files_only=True,dtype=torch.float16,low_cpu_mem_usage=True).to("cuda").eval()
asr=pipeline("automatic-speech-recognition",model=model,tokenizer=processor.tokenizer,feature_extractor=processor.feature_extractor,dtype=torch.float16,device=0,chunk_length_s=30,return_timestamps=True)
checkpoint=OUTPUT_ROOT/"asr_transcripts_v11.jsonl"; completed={}
if checkpoint.is_file():
    for line in checkpoint.read_text(encoding="utf-8").splitlines():
        row=json.loads(line); completed[row["video_id"]]=row
for index,video_id in enumerate(sorted(video_parts)):
    if video_id in completed: continue
    assets=resolve_assets(RAW_INPUT,video_id,video_parts,keyframe_parts)
    media=json.loads(assets.metadata.read_text()) if assets.metadata.is_file() else {}
    fps=float(media.get("fps") or media.get("frame_rate") or 25.0)
    try: row=transcribe_video(video_id,assets.video,fps,asr)
    except Exception as error: row={"video_id":video_id,"status":"ASR_FAILED","error":f"{type(error).__name__}: {error}"}
    with checkpoint.open("a",encoding="utf-8",newline="\n") as handle: handle.write(json.dumps(row,ensure_ascii=False)+"\n")
    completed[video_id]=row
    if (index+1)%25==0: print({"asr_progress":index+1,"total":len(video_parts)})
del asr,model,processor; torch.cuda.empty_cache()
ASR_ROWS=[completed[key] for key in sorted(completed)]; ASR_INDEX=lexical_index([row for row in ASR_ROWS if row.get("status")=="PASS"])
(OUTPUT_ROOT/"asr_lexical_index_v11.json").write_text(json.dumps(ASR_INDEX,ensure_ascii=False,sort_keys=True)+"\n")


In [ ]:
success=[row for row in ASR_ROWS if row.get("status")=="PASS"]
nonempty=[row for row in success if any(segment.get("normalized_text") for segment in row.get("segments",[]))]
failures=[row for row in ASR_ROWS if row.get("status")!="PASS"]
if len(ASR_ROWS)!=len(video_parts): raise RuntimeError("ASR_AUDIO_AUDIT_COUNT_MISMATCH")
if len(success)/max(len(ASR_ROWS),1)<0.95: raise RuntimeError({"ASR_SUCCESS_GATE_FAIL":len(success),"failures":failures[:20]})
if not ASR_INDEX or len(nonempty)<100: raise RuntimeError("ASR_LEXICAL_OR_NONEMPTY_GATE_FAIL")
import random
random.Random(20260819).shuffle(nonempty)
print([{"video_id":row["video_id"],"excerpt":row["segments"][0]["normalized_text"][:160]} for row in nonempty[:20]])
ASR_STATUS={"status":"PASS","audited":len(ASR_ROWS),"successful":len(success),"nonempty":len(nonempty),"lexical_terms":len(ASR_INDEX),"failures":len(failures)}


In [ ]:
# Queries and immutable B0 are prediction-side only; no gt.jsonl is opened.
from triage_eg.fs1.io import read_jsonl
from triage_eg.fs1.runner import group_predictions
from triage_eg.fs1_v11.events import compile_query_events
from triage_eg.fs1.router import route_events,route_query
QUERIES={"cross":read_jsonl(BENCH["dev_cross_60"]/"queries.jsonl"),"l21":read_jsonl(BENCH["dev_l21_150"]/"queries.jsonl")}
B0_PATHS={"cross":PREP/"frozen_baseline/cross_b0_bcf1_f1.jsonl","l21":PREP/"frozen_baseline/l21_b0_bcf1_f1.jsonl"}
B0={key:read_jsonl(path) for key,path in B0_PATHS.items()}


In [ ]:
# Mandatory XCLIP official 8-frame contract and candidate-local per-event evidence.
from triage_eg.fs1_v11.xclip import XClipAdapter,uniform_indices
from triage_eg.video import OpenCVRawVideoDecoder
xclip=XClipAdapter(XCLIP_ROOT); xclip.load(); XCLIP_ROWS=[]; XCLIP_SMOKE=[]
for benchmark in ("cross","l21"):
    grouped=group_predictions(B0[benchmark])
    for query in QUERIES[benchmark]:
        events=compile_query_events(query,grouped[str(query["query_id"])]); routes=route_events(str(query["task"]),[e.event_text for e in events],available={"action"})
        for event,route in zip(events,routes,strict=True):
            if "action" not in route.modalities and str(query["task"]).upper()!="TRAKE": continue
            for row in grouped[str(query["query_id"])][:3]:
                assets=resolve_assets(RAW_INPUT,str(row["video_id"]),video_parts,keyframe_parts); decoder=OpenCVRawVideoDecoder(str(row["video_id"]),assets.video)
                anchor=int(row.get("frame_id",row.get("frame_ids",[0])[min(event.event_index,len(row.get("frame_ids",[0]))-1)])); radius=max(1,int(decoder.info.fps*3)); indices=uniform_indices(max(0,anchor-radius),min(decoder.info.total_frames-1,anchor+radius))
                frames=[item.image for item in decoder.decode_indices(indices)]; decoder.close(); result=xclip.score(event.event_text,frames)
                XCLIP_ROWS.append({"benchmark":benchmark,"query_id":str(query["query_id"]),"event_index":event.event_index,"video_id":str(row["video_id"]),"start_frame":indices[0],"end_frame":indices[-1],"anchor_frame":anchor,"frame_id":anchor,"rank":int(row["rank"]),"source":"action","provenance":{"model_revision":XCLIP_REVISION,**result}})
                if len(XCLIP_SMOKE)<3: XCLIP_SMOKE.append(result)
xclip.unload()
(OUTPUT_ROOT/"xclip_evidence_v11.jsonl").write_text("".join(json.dumps(row,ensure_ascii=False)+"\n" for row in XCLIP_ROWS))
if len(XCLIP_SMOKE)<3 or not all(row["finite"] for row in XCLIP_SMOKE) or len({row["query_id"] for row in XCLIP_ROWS})<10: raise RuntimeError("XCLIP_HARD_GATE_FAIL")


In [ ]:
# Mandatory Grounding-DINO FP32 official post-processing.
from PIL import Image
from triage_eg.fs1_v11.dino import GroundingDinoAdapter
dino=GroundingDinoAdapter(DINO_ROOT); dino.load(); DINO_ROWS=[]; DINO_SMOKE=[]
for benchmark in ("cross","l21"):
    grouped=group_predictions(B0[benchmark])
    for query in QUERIES[benchmark]:
        text=str(query.get("query","")); route=route_query(str(query["task"]),text,available={"object"})
        if "object" not in route.modalities: continue
        for row in grouped[str(query["query_id"])][:3]:
            assets=resolve_assets(RAW_INPUT,str(row["video_id"]),video_parts,keyframe_parts); decoder=OpenCVRawVideoDecoder(str(row["video_id"]),assets.video); frame_id=int(row.get("frame_id",row.get("frame_ids",[0])[0])); decoded=decoder.decode_indices([frame_id])[0]; decoder.close(); image=Image.fromarray(decoded.image)
            detections=dino.detect(image,"person. car. bag. table. screen.")
            if detections: DINO_ROWS.append({"benchmark":benchmark,"query_id":str(query["query_id"]),"video_id":str(row["video_id"]),"frame_id":frame_id,"rank":int(row["rank"]),"source":"object","detections":detections,"provenance":{"model_revision":DINO_REVISION}})
            if len(DINO_SMOKE)<5: DINO_SMOKE.append({"size":image.size,"detections":detections})
dino.unload(); torch.cuda.empty_cache()
(OUTPUT_ROOT/"dino_evidence_v11.jsonl").write_text("".join(json.dumps(row,ensure_ascii=False)+"\n" for row in DINO_ROWS))
if len(DINO_SMOKE)<5 or not any(row["detections"] for row in DINO_SMOKE) or not DINO_ROWS: raise RuntimeError("DINO_HARD_GATE_FAIL")


In [ ]:
# PP-OCR isolated runtime. Full-corpus is not claimed; working local evidence is mandatory.
import venv
from triage_eg.fs1_v11.ocr import isolated_install_commands,run_install
OCR_VENV=Path("/kaggle/working/ppocr_v11_venv"); venv.EnvBuilder(with_pip=True,clear=True).create(OCR_VENV); ocr_python=OCR_VENV/"bin/python"
attempts=[]
for command in isolated_install_commands(ocr_python):
    result=run_install(command); attempts.append(result)
    if result["returncode"]==0: break
if not attempts or attempts[-1]["returncode"]!=0: raise RuntimeError({"PPOCR_BOTH_ISOLATED_INSTALL_ATTEMPTS_FAILED":attempts})
# Worker receives only bounded candidate images; Paddle remains outside the main environment.
worker=Path("/kaggle/working/ppocr_v11_worker.py")
worker.write_text('''import json,sys
from paddleocr import PaddleOCR
ocr=PaddleOCR(lang="vi",ocr_version="PP-OCRv5",device="gpu")
rows=[]
for item in json.loads(open(sys.argv[1],encoding="utf-8").read()):
 try:
  output=ocr.predict(item["path"])
  text=" ".join(str(output))
  if text.strip(): rows.append({**item,"raw_text":text[:1000],"normalized_text":" ".join(text.split()),"confidence":None})
 except Exception as error: rows.append({**item,"status":"OCR_FAILED","error":f"{type(error).__name__}: {error}"})
open(sys.argv[2],"w",encoding="utf-8").write("".join(json.dumps(x,ensure_ascii=False)+"\n" for x in rows))
''',encoding="utf-8")
ocr_inputs=[]; image_root=Path("/kaggle/working/ocr_local_images"); image_root.mkdir(exist_ok=True)
for benchmark in ("cross","l21"):
    grouped=group_predictions(B0[benchmark])
    for query in QUERIES[benchmark]:
        if "ocr" not in route_query(str(query["task"]),str(query.get("query","")),available={"ocr"}).modalities: continue
        for row in grouped[str(query["query_id"])][:10]:
            assets=resolve_assets(RAW_INPUT,str(row["video_id"]),video_parts,keyframe_parts); decoder=OpenCVRawVideoDecoder(str(row["video_id"]),assets.video); frame_id=int(row.get("frame_id",row.get("frame_ids",[0])[0])); image=decoder.decode_indices([frame_id])[0].image; decoder.close(); path=image_root/f"{benchmark}_{query['query_id']}_{row['rank']}.jpg"; Image.fromarray(image).save(path); ocr_inputs.append({"benchmark":benchmark,"query_id":str(query["query_id"]),"video_id":str(row["video_id"]),"actual_frame_id":frame_id,"frame_id":frame_id,"rank":int(row["rank"]),"source":"ocr","path":str(path)})
input_json=Path("/kaggle/working/ocr_inputs.json"); input_json.write_text(json.dumps(ocr_inputs)); ocr_output=OUTPUT_ROOT/"ocr_records_v11.jsonl"; process=subprocess.run([str(ocr_python),str(worker),str(input_json),str(ocr_output)],capture_output=True,text=True)
if process.returncode: raise RuntimeError({"PPOCR_WORKER_FAILED":process.stderr.splitlines()[-30:],"attempts":attempts})
OCR_ROWS=[json.loads(line) for line in ocr_output.read_text(encoding="utf-8").splitlines() if line.strip() and '"normalized_text"' in line]
OCR_INDEX={}
import re
for row in OCR_ROWS:
    for token in set(re.findall(r"\w+",row["normalized_text"].casefold())): OCR_INDEX.setdefault(token,[]).append({"video_id":row["video_id"],"frame_id":row["frame_id"],"text":row["normalized_text"]})
(OUTPUT_ROOT/"ocr_lexical_index_v11.json").write_text(json.dumps(OCR_INDEX,ensure_ascii=False,sort_keys=True)+"\n")
if not OCR_ROWS or not OCR_INDEX: raise RuntimeError("OCR_LOCAL_ONLY_HARD_GATE_FAIL")
OCR_STATUS={"status":"OCR_LOCAL_ONLY","processed":len(ocr_inputs),"nonempty_spans":len(OCR_ROWS),"lexical_terms":len(OCR_INDEX),"attempts":attempts}


In [ ]:
PLUGIN_STATUS={"asr":ASR_STATUS,"xclip":{"status":"PASS","records":len(XCLIP_ROWS),"smoke":XCLIP_SMOKE},"dino":{"status":"PASS","records":len(DINO_ROWS),"smoke":DINO_SMOKE},"ocr":OCR_STATUS,"sam":{"status":"PASS_RETAINED_FROM_DIAGNOSIS"},"qwen":{"status":"PINNED_ASSET_VALIDATED","asset":str(QWEN_ROOT)}}
EVIDENCE_MANIFEST={"version":"TRIAGEEG_COMPLETION_V11","source_dataset":str(RAW_INPUT),"gt_access":False,"asr_records":len(ASR_ROWS),"asr_lexical_terms":len(ASR_INDEX),"xclip_records":len(XCLIP_ROWS),"dino_records":len(DINO_ROWS),"ocr_mode":OCR_STATUS["status"],"ocr_records":len(OCR_ROWS),"plugin_status":PLUGIN_STATUS}
for name,value in (("plugin_status.json",PLUGIN_STATUS),("evidence_manifest.json",EVIDENCE_MANIFEST),("tests_summary.json",TEST_SUMMARY),("diagnosis_freeze.json",DIAGNOSIS)): (OUTPUT_ROOT/name).write_text(json.dumps(value,indent=2,default=str)+"\n")
import shutil
shutil.make_archive(str(OUTPUT_ZIP.with_suffix("")),"zip",OUTPUT_ROOT)
print({"download_zip":str(OUTPUT_ZIP),"manifest":EVIDENCE_MANIFEST})
